# CyberSec-FT-LLM — Unsloth QLoRA Fine-Tuning
Fine-tunes **Phi-3.5-mini-Instruct** on CVE/Exploit security data.

**Before running:**
1. Runtime → Change runtime type → **GPU (T4 or A100)**
2. Upload `dataset/` to `My Drive/CyberSec-FT-LLM/dataset/`
3. Run all cells in order ▶

In [ ]:
# Cell 1 — Install
!pip install -q unsloth trl>=0.9.0 transformers>=4.45.0 datasets accelerate peft bitsandbytes pyyaml rouge-score
print('Done.')

In [ ]:
# Cell 2 — GPU Check
import torch
print(f'CUDA : {torch.cuda.is_available()}')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 3 — Mount Drive & Verify Dataset
from google.colab import drive
drive.mount('/content/drive')
import os
DATASET_DIR = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    p = os.path.join(DATASET_DIR, f)
    print(f'  {f}: {os.path.getsize(p)/1e6:.1f}MB' if os.path.exists(p) else f'  {f}: NOT FOUND!')

In [ ]:
# Cell 4 — Clone Repo
import os
REPO_URL = 'https://github.com/Mohamedabul/CyberSec-FT-LLM.git'
REPO_DIR = '/content/CyberSec-FT-LLM'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('Repo ready. Contents:', os.listdir('.'))

In [ ]:
# Cell 5 — Link Dataset from Drive
import os
SRC = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
DST = '/content/CyberSec-FT-LLM/dataset'
os.makedirs(DST, exist_ok=True)
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    dst = os.path.join(DST, f)
    if os.path.exists(dst) or os.path.islink(dst): os.remove(dst)
    src = os.path.join(SRC, f)
    if os.path.exists(src):
        os.symlink(src, dst); print(f'  Linked: {f}')
    else:
        print(f'  MISSING: {f}')

In [ ]:
# Cell 6 — Configure Paths
import yaml, os
config_path = '/content/CyberSec-FT-LLM/configs/training_config.yaml'
if not os.path.exists(config_path):
    raise FileNotFoundError(f'Not found: {config_path} — did Cell 4 clone successfully?')
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg['colab']['output_dir']   = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
cfg['colab']['dataset_path'] = '/content/CyberSec-FT-LLM/dataset'
with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
os.makedirs(cfg['colab']['output_dir'], exist_ok=True)
print('Config updated:')
print(f"  output_dir  : {cfg['colab']['output_dir']}")
print(f"  dataset_path: {cfg['colab']['dataset_path']}")

## Cell 7 — Training
You will see **live output** at every step:
```
{'loss': 1.432, 'grad_norm': 0.85, 'learning_rate': 1.9e-4, 'epoch': 0.01, 'step': 25}
{'loss': 1.281, 'grad_norm': 0.71, 'learning_rate': 1.8e-4, 'epoch': 0.03, 'step': 50}
```
- Checkpoints auto-saved to Google Drive every **200 steps**
- If Colab disconnects → just **re-run this cell** — it auto-resumes

In [ ]:
# Cell 7 — Inline Training (live output)
import json, yaml, glob, os, sys
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

sys.path.insert(0, '/content/CyberSec-FT-LLM/training/unsloth')
from model_loader_unsloth import load_model_and_tokenizer

BASE = Path('/content/CyberSec-FT-LLM')
def load_yaml(n): return yaml.safe_load(open(BASE/'configs'/n))

model_cfg = load_yaml('model_config.yaml')
train_cfg = load_yaml('training_config.yaml')
tc, cc = train_cfg['common'], train_cfg['colab']

cfg = {
    'model':        {**model_cfg['base_model'], 'max_seq_length': model_cfg['tokenizer']['max_length']},
    'quantization': model_cfg['quantization'],
    'lora':         model_cfg['lora'],
}

DATASET_BASE = Path(cc['dataset_path'])
adapter_dir  = cc['output_dir']

def load_jsonl(path):
    rows = []
    for line in open(path, encoding='utf-8', errors='replace'):
        line = line.replace('\x00','').strip()
        if line:
            try: rows.append(json.loads(line))
            except: pass
    return rows

def fmt(s, tok):
    u = f"{s.get('instruction','')}\n\n{s.get('input','')}" if s.get('input') else s.get('instruction','')
    return tok.apply_chat_template(
        [{'role':'user','content':u},{'role':'assistant','content':s.get('output','')}],
        tokenize=False, add_generation_prompt=False)

print('='*55)
print('STEP 1/5 — Loading model (4-bit Unsloth)...')
print('='*55)
model, tokenizer = load_model_and_tokenizer(cfg)

print('\n'+'='*55)
print('STEP 2/5 — Loading datasets...')
print('='*55)
train_raw = load_jsonl(str(DATASET_BASE/'train.jsonl'))
val_raw   = load_jsonl(str(DATASET_BASE/'val.jsonl'))
print(f'  Train: {len(train_raw):,} | Val: {len(val_raw):,}')
train_ds = Dataset.from_list([{'text': fmt(s, tokenizer)} for s in train_raw])
val_ds   = Dataset.from_list([{'text': fmt(s, tokenizer)} for s in val_raw])
del train_raw, val_raw

print('\n'+'='*55)
print('STEP 3/5 — Configuring trainer...')
print('='*55)
os.makedirs(adapter_dir, exist_ok=True)
args = SFTConfig(
    output_dir=adapter_dir,
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=cc['per_device_train_batch_size'],
    gradient_accumulation_steps=cc['gradient_accumulation_steps'],
    learning_rate=tc['learning_rate'],
    lr_scheduler_type=tc['lr_scheduler_type'],
    warmup_steps=tc['warmup_steps'],
    weight_decay=tc['weight_decay'],
    bf16=cc['bf16'],
    gradient_checkpointing=tc['gradient_checkpointing'],
    optim=cc['optim'],
    logging_steps=tc['logging_steps'],
    save_steps=tc['save_steps'],
    eval_steps=tc['eval_steps'],
    eval_strategy='steps',
    save_total_limit=tc['save_total_limit'],
    load_best_model_at_end=tc['load_best_model_at_end'],
    report_to=tc['report_to'],
    seed=tc['seed'],
    dataset_text_field='text',
)
trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds, args=args,
)

checkpoints = sorted(glob.glob(os.path.join(adapter_dir, 'checkpoint-*')))
resume_from = checkpoints[-1] if checkpoints else None

print('\n'+'='*55)
print('STEP 4/5 — Training')
print(f'  Resume : {os.path.basename(resume_from) if resume_from else "Fresh start"}')
print(f'  Steps  : {len(trainer.get_train_dataloader()) // cc["gradient_accumulation_steps"]} total')
print('  Log    : loss | grad_norm | lr | epoch — every 25 steps')
print('  Save   : checkpoint to Drive every 200 steps')
print('='*55)
trainer.train(resume_from_checkpoint=resume_from)

print('\n'+'='*55)
print('STEP 5/5 — Saving adapter + merging...')
print('='*55)
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'  Adapter saved: {adapter_dir}')

from unsloth import FastLanguageModel
merged_dir = adapter_dir.replace('/adapter', '/merged')
merged, mtok = FastLanguageModel.from_pretrained(
    model_name=adapter_dir, max_seq_length=cfg['model']['max_seq_length'],
    dtype=None, load_in_4bit=True)
os.makedirs(merged_dir, exist_ok=True)
merged.save_pretrained_merged(merged_dir, mtok, save_method='merged_16bit')
print(f'  Merged model: {merged_dir}')
print('\nAll done! Model saved to Google Drive.')

In [ ]:
# Cell 8 — Quick Inference Test
import sys, torch
sys.path.insert(0, '/content/CyberSec-FT-LLM/training/unsloth')
from model_loader_unsloth import load_for_inference

ADAPTER = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
model, tokenizer = load_for_inference(ADAPTER, max_seq_length=512)

instruction = 'Analyze the following CVE and provide a structured vulnerability report.'
context = 'CVE ID: CVE-2021-44228\nDescription: Apache Log4j2 JNDI RCE (Log4Shell).\nCVSS v3: 10.0 CRITICAL | CWE: CWE-917'
msgs = [{'role':'user','content':f'{instruction}\n\n{context}'}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# Cell 9 — Verify Saved Files
import os
for folder in [
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter',
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/merged',
]:
    if os.path.exists(folder):
        files = os.listdir(folder)
        size  = sum(os.path.getsize(os.path.join(folder,f)) for f in files)/1e6
        print(f'  {folder}\n    {len(files)} files | {size:.0f} MB')
    else:
        print(f'  Not found: {folder}')